# 전처리

In [ ]:
# 1. 나눔글꼴 설치
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-nanum is already the newest version (20200506-1).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.
/usr/share/fonts: caching, new cache contents: 0 fonts, 1 dirs
/usr/share/fonts/truetype: caching, new cache contents: 0 fonts, 3 dirs
/usr/share/fonts/truetype/humor-sans: caching, new cache contents: 1 fonts, 0 dirs
/usr/share/fonts/truetype/liberation: caching, new cache contents: 16 fonts, 0 dirs
/usr/share/fonts/truetype/nanum: caching, new cache contents: 12 fonts, 0 dirs
/usr/local/share/fonts: caching, new cache contents: 0 fonts, 0 dirs
/root/.local/share/fonts: skipping, no such directory
/root/.fonts: skipping, no such directory
/usr/share/fonts/truetype: skipping, looped directory detected
/usr/share/fonts/truetype/humor-sans: skipping, looped directory detected
/usr/share/fonts/truetype/liberation: skipping, looped directory detected
/usr/share/fonts/truetype/

In [ ]:
# 2. 글꼴 설정 및 확인
import matplotlib.pyplot as plt
import matplotlib as mpl

# 글꼴을 나눔고딕으로 설정
plt.rc('font', family='NanumGothic')

# 마이너스 기호 깨짐 방지
mpl.rcParams['axes.unicode_minus'] = False

In [ ]:
import pandas as pd
import os

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Purchase (매입 데이터)


타겟 변수: **quantity**

### 1. 시간 피처 생성

In [ ]:
purchase=pd.read_csv('/content/drive/MyDrive/프로젝트 준비/전처리/B_purchase_1.xls')
purchase['date']=pd.to_datetime(purchase['date'])

# quantity 또는 supply_price 가 음수이거나 0인 행 제거
df_cleaned = purchase[(purchase["quantity"] >= 0) & (purchase["sales_price"] >= 0)].copy()

# 결과 확인
print("원본 행 개수:", len(purchase))
print("정제 후 행 개수:", len(df_cleaned))

#판매수량이나 가격이 음수이인 행 제거
purchase=purchase[(purchase["quantity"] >= 0) & (purchase["sales_price"] >= 0)]

원본 행 개수: 1915
정제 후 행 개수: 1915


In [ ]:
######quarter 변수 생성######

# month 기준으로 quarter 변수 생성
purchase["quarter"] = ((purchase["month"] - 1) // 3) + 1

######holiday: 주말 또는 공휴일이면 1, 아니면 0#######
import holidays

# 1날짜 범위 생성
date_range = pd.date_range(start="2021-01-01", end="2025-03-31", freq="D")
df = pd.DataFrame({"date": date_range})

# 한국 공휴일 세팅
kr_holidays = holidays.KR(years=[2021, 2022, 2023, 2024, 2025])

# 공휴일 여부
df["is_holiday"] = df["date"].isin(kr_holidays)

# 주말 여부 (토:5, 일:6)
df["is_weekend"] = df["date"].dt.weekday >= 5

#  holiday 변수 생성 (공휴일 or 주말 → 1, 그 외 → 0)
df["holiday"] = ((df["is_holiday"]) | (df["is_weekend"])).astype(int)

df = df[["date", "holiday"]]

print(df.head(15))

# holiday df와 원본 데이터 merge
purchase=purchase.merge(df[["date", "holiday"]], on="date", how="left")
purchase.head()

         date  holiday
0  2021-01-01        1
1  2021-01-02        1
2  2021-01-03        1
3  2021-01-04        0
4  2021-01-05        0
5  2021-01-06        0
6  2021-01-07        0
7  2021-01-08        0
8  2021-01-09        1
9  2021-01-10        1
10 2021-01-11        0
11 2021-01-12        0
12 2021-01-13        0
13 2021-01-14        0
14 2021-01-15        0


/tmp/ipython-input-530667577.py:17: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  df["is_holiday"] = df["date"].isin(kr_holidays)


,work_type,date,client_id,client_zipcode,supplier_code,supplier_zipcode,stock_in_type,product_code,barcode,product_name,...,sales_price,vat,main_category,mid_category,sub_category,year,month,day,quarter,holiday
0,입고,2021-01-05,NaN,NaN,100004,51402,센터에서 정상적으로 입고,8801056103569,8.801056e+12,(롯데칠성)트레비금귤500ml,...,203800,20380,"음료,차류",음료,탄산음료,2021,1,5,1,0
1,입고,2021-01-05,NaN,NaN,100004,51402,센터에서 정상적으로 입고,8801056143015,8.801056e+12,(롯데칠성)칠성사이다250ml,...,2250000,225000,"음료,차류",음료,탄산음료,2021,1,5,1,0
2,입고,2021-01-05,NaN,NaN,100010,50859,센터에서 정상적으로 입고,8801094017606,8.801094e+12,(코카콜라)코카콜라500ml,...,1374900,137490,"음료,차류",음료,탄산음료,2021,1,5,1,0
3,입고,2021-01-05,NaN,NaN,100010,50859,센터에서 정상적으로 입고,8801094252403,8.801094e+12,(코카콜라)밀크소다1.5L(암바사),...,159000,15900,"음료,차류",음료,탄산음료,2021,1,5,1,0
4,입고,2021-01-05,NaN,NaN,300023,41142,센터에서 정상적으로 입고,8801043034722,8.801043e+12,(농심)웰치스소다포도1.5L,...,98100,9810,"음료,차류",음료,탄산음료,2021,1,5,1,0


In [ ]:
# 칼럼 삭제
purchase.drop(['client_id', 'client_zipcode',
       'supplier_zipcode', 'stock_in_type', 'barcode', 'spec',
        'option_code', 'option_name', 'units_per_pack',
        'each_count','vat', 'main_category',
       'mid_category', 'sub_category'], axis=1, inplace=True)

## Sales(매출데이터)


타겟 변수: **quantity**

### 1. 시간 피처 생성

In [ ]:
sales=pd.read_csv('/content/drive/MyDrive/프로젝트 준비/전처리/B_sales_1.xls')

sales['sales_date']=pd.to_datetime(sales['sales_date'])

# quantity 또는 supply_price 가 음수이거나 0인 행 제거
df_cleaned = sales[(sales["quantity"] >= 0) & (sales["supply_price"] >= 0)].copy()

# 결과 확인
print("원본 행 개수:", len(sales))
print("정제 후 행 개수:", len(df_cleaned))

#판매수량이나 가격이 음수이인 행 제거
sales=sales[(sales["quantity"] >= 0) & (sales["supply_price"] >= 0)]

원본 행 개수: 28299
정제 후 행 개수: 28240


In [ ]:
##### month 기준으로 quarter 변수 생성######
sales["quarter"] = ((sales["month"] - 1) // 3) + 1

In [ ]:
#칼럼 삭제
sales.drop(['spec','units_per_pack','client_zipcode',
            'client_id','option_code','main_category','mid_category',
            'sub_category','vat'], axis=1, inplace=True)

##2. 과거 데이터 기반 피처 생성

### 1) 매입 데이터 df용

In [ ]:
import re
import pandas as pd
import numpy as np
# =========================
# Step 1) 탄산음료 파생변수
# =========================
def split_name(name: str):
    maker = re.findall(r"\((.*?)\)", str(name))
    maker = maker[0] if maker else None
    cleaned = re.sub(r"\(.*?\)", "", str(name))
    product = re.split(r"\d", cleaned)[0]
    return maker, product.strip()

# 제조사(company), 제로 여부
purchase[["company", "name"]] = purchase["product_name"].apply(lambda x: pd.Series(split_name(x)))

# =========================
# Step 2) name × 연월 집계
# =========================
PURCHASE_DATE_COL = "date"
QTY_COL           = "quantity"
PRICE_COL         = "sales_price"
GROUP_KEY_COLS    = ["supplier_code"]

def _ensure_datetime(df, date_col):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    return df

def _make_key(df, key_cols):
    if len(key_cols) == 1:
        return df[key_cols[0]].astype(str)
    return df[key_cols].astype(str).agg("§".join, axis=1)

# --- 기본 전처리
purchase = _ensure_datetime(purchase, PURCHASE_DATE_COL)
purchase["_grp_key"] = _make_key(purchase, GROUP_KEY_COLS)
purchase["_ym"] = purchase[PURCHASE_DATE_COL].dt.to_period("M").astype(str)

# --- 연월 기준 집계
monthly = (
    purchase.groupby(["_grp_key", "_ym"], as_index=False)
    .agg({
        QTY_COL   : "sum",   # 월별 총 수량
        PRICE_COL : "sum",   # 월별 총 금액
        "holiday" : "sum",   # 월별 주말/공휴일 주문 건수
    })
    .rename(columns={
        QTY_COL: "quantity",            # inbound_qty → quantity
        PRICE_COL: "sales_price",       # 매입금액 합계
        "holiday": "holiday_count",
    })
)

# =========================
# Step 3) 시간 변수 & 정렬
# =========================
monthly["_ym_period"] = pd.to_datetime(monthly["_ym"] + "-01")
monthly = monthly.sort_values(["_grp_key", "_ym_period"])

monthly["year"] = monthly["_ym"].str[:4].astype(int)
monthly["month"] = monthly["_ym"].str[5:7].astype(int)
monthly["quarter"] = ((monthly["month"] - 1) // 3) + 1

# =========================
# Step 4) lag & rolling
# =========================
def _lag_roll(g, window=3):
    g = g.copy()
    g["sales_lag_1m"] = g["quantity"].shift(1)
    g["sales_lag_1y"] = g["quantity"].shift(12)
    g["sales_rolling_avg_3m"] = g["quantity"].rolling(window).mean()
    g["sales_rolling_max_3m"] = g["quantity"].rolling(window).max()
    g["sales_rolling_min_3m"] = g["quantity"].rolling(window).min()
    return g

monthly = monthly.groupby("_grp_key", group_keys=False).apply(_lag_roll, window=3)

# =========================
# Step 5) 단가 & 가격 변동성
# =========================
monthly["unit_price"] = monthly["sales_price"] / monthly["quantity"].replace(0, np.nan)

def _price_vol(g, window=3):
    g = g.copy()
    g["price_volatility"] = g["unit_price"].rolling(window=window, min_periods=2).std()
    return g

monthly = monthly.groupby("_grp_key", group_keys=False).apply(_price_vol, window=3)

# =========================
# Step 6) 최종 정리
# =========================

monthly = monthly.drop(columns=["_ym_period"]).rename(columns={"_grp_key": "supplier_code"})


monthly = monthly.sort_values(["year", "month"]).reset_index(drop=True)


purchase["supplier_code"] = purchase["supplier_code"].astype(str)
monthly["supplier_code"]  = monthly["supplier_code"].astype(str)

rep_map = (
    purchase.groupby(["supplier_code", "product_name"])
            .size()
            .reset_index(name="cnt")
            .sort_values(["supplier_code", "cnt"], ascending=[True, False])
            .drop_duplicates("supplier_code")[["supplier_code", "product_name"]]
)

monthly = monthly.merge(rep_map, on="supplier_code", how="left", validate="m:1")

monthly = monthly[
    ["supplier_code", "product_name", "year", "month", "quarter",
     "quantity", "sales_price", "holiday_count",
     "sales_lag_1m", "sales_lag_1y",
     "sales_rolling_avg_3m", "sales_rolling_max_3m", "sales_rolling_min_3m",
     "unit_price", "price_volatility"]
]


# ===========탄산 음료 추가 파생변수=========
# 제로 유무 추가
monthly["is_zero"] = monthly["product_name"].str.contains("제로", na=False).astype(int)

# 음료 용량에 관해서
def extract_volume(name):
    matches = re.findall(r"(\d+[.,]?\d*)(ml|ML|l|L|ℓ)?", name)
    if matches:
        val, unit = matches[-1]  # 마지막 매칭
        val = float(val.replace(",", "."))  # 1,5 → 1.5

        if not unit:
            val *= 1000
        elif unit.lower() in ["l", "ℓ"]:
            val *= 1000

        return val
    return np.nan

monthly["volume_ml"] = monthly["product_name"].apply(extract_volume)

monthly_purchase=monthly.copy()
monthly_purchase

/tmp/ipython-input-147627735.py:77: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  monthly = monthly.groupby("_grp_key", group_keys=False).apply(_lag_roll, window=3)
/tmp/ipython-input-147627735.py:89: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  monthly = monthly.groupby("_grp_key", group_keys=False).apply(_price_vol, window=3)


,supplier_code,product_name,year,month,quarter,quantity,sales_price,holiday_count,sales_lag_1m,sales_lag_1y,sales_rolling_avg_3m,sales_rolling_max_3m,sales_rolling_min_3m,unit_price,price_volatility,is_zero,volume_ml
0,100002,(동아오츠카)데미소다애플250ml,2021,1,1,300,109000,0,NaN,NaN,NaN,NaN,NaN,363.333333,NaN,0,250.0
1,100004,(롯데칠성)칠성사이다250ml,2021,1,1,6992,5609500,0,NaN,NaN,NaN,NaN,NaN,802.274027,NaN,0,250.0
2,100010,(코카콜라)코카콜라1.5L,2021,1,1,4806,6458800,0,NaN,NaN,NaN,NaN,NaN,1343.903454,NaN,0,1500.0
3,100048,(일화)맥콜1.5L,2021,1,1,940,827100,0,NaN,NaN,NaN,NaN,NaN,879.893617,NaN,0,1500.0
4,100088,(농심)웰치스청포도355ml,2021,1,1,480,781900,0,NaN,NaN,NaN,NaN,NaN,1628.958333,NaN,0,355.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
402,100010,(코카콜라)코카콜라1.5L,2024,12,4,792,662100,0,1878.0,1698.0,1756.000000,2598.0,792.0,835.984848,167.141622,0,1500.0
403,100112,(롯데칠성)펩시콜라1.5L,2024,12,4,390,529300,0,1520.0,1440.0,796.666667,1520.0,390.0,1357.179487,275.629310,0,1500.0
404,300022,(일화)맥콜캔250ml,2024,12,4,2300,2107100,0,2860.0,3960.0,4116.000000,7188.0,2300.0,916.130435,91.312331,0,250.0
405,300023,(코카콜라)코카콜라(슈퍼용)355ml,2024,12,4,300,173600,0,720.0,1260.0,2280.000000,5820.0,300.0,578.666667,256.084976,0,355.0


### 2) 매출 데이터 df2 용 (판매 중심)

In [ ]:
# =========================
# Step 1) 탄산음료 파생변수
# =========================
import re
import pandas as pd
import numpy as np

def split_name(name: str):
    maker = re.findall(r"\((.*?)\)", str(name))
    maker = maker[0] if maker else None
    cleaned = re.sub(r"\(.*?\)", "", str(name))
    product = re.split(r"\d", cleaned)[0]
    return maker, product.strip()

# 제조사(company), 제로 여부
sales[["company", "name"]] = sales["product_name"].apply(lambda x: pd.Series(split_name(x)))

# =========================
# Step 2) name × 연월 집계
# =========================
SALES_DATE_COL = "sales_date"
QTY_COL           = "quantity"
PRICE_COL         = "supply_price"
GROUP_KEY_COLS    = ["barcode"]

def _ensure_datetime(df, date_col):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    return df

def _make_key(df, key_cols):
    if len(key_cols) == 1:
        return df[key_cols[0]].astype(str)
    return df[key_cols].astype(str).agg("§".join, axis=1)

# --- 기본 전처리
sales = _ensure_datetime(sales, SALES_DATE_COL)
sales["_grp_key"] = _make_key(sales, GROUP_KEY_COLS)
sales["_ym"] = sales[SALES_DATE_COL].dt.to_period("M").astype(str)

# --- 연월 기준 집계
monthly_sales = (
    sales.groupby(["_grp_key", "_ym"], as_index=False)
    .agg({
        QTY_COL   : "sum",   # 월별 총 수량
        PRICE_COL : "sum",   # 월별 총 금액
    })
    .rename(columns={
        QTY_COL: "quantity",            # inbound_qty → quantity
        PRICE_COL: "supply_price",       # 매입금액 합계
    })
)

# =========================
# Step 3) 시간 변수 & 정렬
# =========================
monthly_sales["_ym_period"] = pd.to_datetime(monthly_sales["_ym"] + "-01")
monthly_sales = monthly_sales.sort_values(["_grp_key", "_ym_period"])

monthly_sales["year"] = monthly_sales["_ym"].str[:4].astype(int)
monthly_sales["month"] = monthly_sales["_ym"].str[5:7].astype(int)
monthly_sales["quarter"] = ((monthly_sales["month"] - 1) // 3) + 1

# =========================
# Step 4) lag & rolling
# =========================
def _sales_lag_roll(g, window=3):
    g = g.copy()
    g["supply_price"] = g["supply_price"].fillna(0)
    g["inbound_qty_lag_1m"] = g["supply_price"].shift(1)
    g["inbound_qty_lag_1y"] = g["supply_price"].shift(12)

    g["inbound_rolling_avg_3m"] = g["supply_price"].rolling(window).mean()
    g["inbound_rolling_max_3m"] = g["supply_price"].rolling(window).max()
    g["inbound_rolling_min_3m"] = g["supply_price"].rolling(window).min()
    return g

monthly_sales = monthly_sales.groupby("_grp_key", group_keys=False).apply(_sales_lag_roll, window=3) # Corrected function name

# =========================
# Step 5) 단가 & 가격 변동성
# =========================
monthly_sales["unit_supply_price"] = monthly_sales["supply_price"] / monthly_sales["quantity"].replace(0, np.nan)

def _price_vol(g, window=3):
    g = g.copy()
    g["price_volatility"] = g["unit_supply_price"].rolling(window=window, min_periods=2).std()
    return g

monthly_sales = monthly_sales.groupby("_grp_key", group_keys=False).apply(_price_vol, window=3)

# =========================
# Step 6) 최종 정리
# ========================


monthly_sales = monthly_sales.drop(columns=["_ym_period"]).rename(columns={"_grp_key": "barcode"})

monthly_sales = monthly_sales.sort_values(["year", "month"]).reset_index(drop=True)


sales["barcode"] = sales["barcode"].astype(str)
monthly_sales["barcode"]  = monthly_sales["barcode"].astype(str)

rep_map = (
    sales.groupby(["barcode", "product_name"])
            .size()
            .reset_index(name="cnt")
            .sort_values(["barcode", "cnt"], ascending=[True, False])
            .drop_duplicates("barcode")[["barcode", "product_name"]]
)

monthly_sales = monthly_sales.merge(rep_map, on="barcode", how="left", validate="m:1")

monthly_sales = monthly_sales[
    ["barcode","product_name", "year", "month", "quarter",
     "quantity", "supply_price",
     "inbound_qty_lag_1m", "inbound_qty_lag_1y",
     "inbound_rolling_avg_3m", "inbound_rolling_max_3m", "inbound_rolling_min_3m",
     "unit_supply_price", "price_volatility"]
]


# ===========탄산 음료 추가 파생변수=========
# 제로 유무 추가
monthly_sales["is_zero"] = monthly_sales["product_name"].str.contains("제로", na=False).astype(int)

# 음료 용량에 관해서
def extract_volume(name):
    matches = re.findall(r"(\d+[.,]?\d*)(ml|ML|l|L|ℓ)?", name)
    if matches:
        val, unit = matches[-1]  # 마지막 매칭
        val = float(val.replace(",", "."))  # 1,5 → 1.5

        if not unit:
            val *= 1000
        elif unit.lower() in ["l", "ℓ"]:
            val *= 1000

        return val
    return np.nan

monthly_sales["volume_ml"] = monthly_sales["product_name"].apply(extract_volume)

monthly_sales

/tmp/ipython-input-689278866.py:78: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  monthly_sales = monthly_sales.groupby("_grp_key", group_keys=False).apply(_sales_lag_roll, window=3) # Corrected function name
/tmp/ipython-input-689278866.py:90: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  monthly_sales = monthly_sales.groupby("_grp_key", group_keys=False).apply(_price_vol, window=3)


,barcode,product_name,year,month,quarter,quantity,supply_price,inbound_qty_lag_1m,inbound_qty_lag_1y,inbound_rolling_avg_3m,inbound_rolling_max_3m,inbound_rolling_min_3m,unit_supply_price,price_volatility,is_zero,volume_ml
0,18801097234229.0,(동아오츠카)오란씨파인1.5L,2021,1,1,2,24000,NaN,NaN,NaN,NaN,NaN,12000.000000,NaN,0,1500.0
1,18801097260020.0,(동아오츠카)데미소다애플1.5L,2021,1,1,7,72800,NaN,NaN,NaN,NaN,NaN,10400.000000,NaN,0,1500.0
2,18801223100268.0,(일화)맥콜페트500ml,2021,1,1,5,80000,NaN,NaN,NaN,NaN,NaN,16000.000000,NaN,0,500.0
3,18801223100275.0,(일화)맥콜1.5L,2021,1,1,38,585600,NaN,NaN,NaN,NaN,NaN,15410.526316,NaN,0,1500.0
4,18801223100503.0,(일화)맥콜캔250ml,2021,1,1,22,269000,NaN,NaN,NaN,NaN,NaN,12227.272727,NaN,0,250.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3386,8801097168800.0,(동아오츠카)데미소다청포도캔250ml,2024,12,4,60,31300,30000.0,135000.0,30433.333333,31300.0,30000.0,521.666667,12.509256,0,250.0
3387,8801223100254.0,(일화)맥콜캔250ml,2024,12,4,60,30000,30000.0,230100.0,95900.000000,227700.0,30000.0,500.000000,80.004252,0,250.0
3388,8801223100261.0,(일화)맥콜페트500ml,2024,12,4,96,83600,20900.0,17400.0,76633.333333,125400.0,20900.0,870.833333,0.000000,0,500.0
3389,8801223100278.0,(일화)맥콜1.5L,2024,12,4,12,18500,18500.0,16200.0,30833.333333,55500.0,18500.0,1541.666667,0.000000,0,1500.0


## 파생 변수 생성

(1) 날씨 관련 파생변수 생성

(2) 경제 관련 파생변수 생성

In [ ]:
#=======날씨 관련 변수 생성========
weather=pd.read_csv('/content/drive/MyDrive/프로젝트 준비/전처리/weather_data.csv')

#=======경제 관련 변수 생성=======

cpi=pd.read_csv('/content/drive/MyDrive/프로젝트 준비/전처리/소비자물가지수.csv')

survey=pd.read_csv('/content/drive/MyDrive/프로젝트 준비/전처리/소비자심리지수.csv', encoding='cp949')
survey=survey[(survey['CSI코드별']=='소비자심리지수')&(survey['CSI분류코드별']=='전체')]

cpi_soda=pd.read_excel('/content/drive/MyDrive/프로젝트 준비/전처리/소비자물가지수_탄산음료.xlsx')
cpi_soda.drop(['지출목적별'], axis=1, inplace=True)


# 소비자심리지수 처리 :ccsi
csi_clean = survey.copy()
csi_clean['시점'] = csi_clean['시점'].str.replace(" 월","",regex=False)
csi_clean[['year','month']] = csi_clean['시점'].str.split('.',expand=True).astype(int)
csi_clean = csi_clean[['year','month','소비자동향조사(전국, 월, 2008.9~)']]
csi_clean.rename(columns={'소비자동향조사(전국, 월, 2008.9~)':'ccsi'}, inplace=True)

# 소비자물가지수 처리 (wide → long 변환) :cpi
cpi_clean = cpi[cpi['시도별']=='전국'].drop(columns=['시도별'])
cpi_long = cpi_clean.melt(var_name='date', value_name='cpi')
cpi_long[['year','month']] = cpi_long['date'].str.split('.',expand=True).astype(int)
cpi_long = cpi_long[['year','month','cpi']]


#소비자 물가지수(탄산음료) :cpi_soda
cpi_soda_clean = cpi_soda[cpi_soda['시도별']=='전국'].drop(columns=['시도별'])
cpi_soda_long = cpi_soda_clean.melt(var_name='date', value_name='cpi_soda')
cpi_soda_long[['year','month']] = cpi_soda_long['date'].str.split('.',expand=True).astype(int)
cpi_soda_long = cpi_soda_long[['year','month','cpi_soda']]

# ===========병합==============

# Step 1: csi + cpi
merged = pd.merge(csi_clean, cpi_long, on=["year","month"], how="outer")

# Step 2: 여기에 cpi_soda까지 병합
merged = pd.merge(merged, cpi_soda_long, on=["year","month"], how="outer")

# Step 3: weather까지
merged = pd.merge(merged, weather, on=["year","month"], how="outer")

# Step 3: 정렬
merged = merged.sort_values(["year","month"]).reset_index(drop=True)

merged.head()

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,year,month,ccsi,cpi,cpi_soda,temperature,rain
0,2021,1,95.2,101.04,103.27,-1.1,19.9
1,2021,2,97.4,101.58,103.79,3.4,20.1
2,2021,3,100.6,101.84,104.72,8.7,110.7
3,2021,4,102.5,101.98,105.91,13.2,76.3
4,2021,5,105.7,102.05,105.22,16.6,143.8


In [ ]:
merged=merged.iloc[:51,:]
merged.tail()

,year,month,ccsi,cpi,cpi_soda,temperature,rain
46,2024,11,100.7,114.40,118.37,9.7,59.6
47,2024,12,88.2,114.91,115.44,1.8,6.5
48,2025,1,91.2,115.71,118.83,-0.2,16.9
49,2025,2,95.2,116.08,120.02,-0.5,15.7
50,2025,3,93.4,116.29,120.35,7.6,48.3


(3) 네이버, 구글 트렌드 관련 변수 생성

In [ ]:
import pandas as pd
# ===========구글 트렌드 변수(탄산음료)================
# 파일 불러오기 (앞 2줄 메타데이터 제거)
google = pd.read_csv("/content/drive/MyDrive/프로젝트 준비/전처리/multiTimeline (1).csv", skiprows=2)

# 날짜 컬럼 확인 후 변환 (보통 '주' 혹은 'Week' 라벨로 되어 있음)
google.rename(columns={google.columns[0]: "date"}, inplace=True)
google["date"] = pd.to_datetime(google["date"], errors="coerce")

# 연, 월 추출
google["year"] = google["date"].dt.year
google["month"] = google["date"].dt.month

# 월별 평균 계산
google_avg = (
    google.groupby(["year", "month"], as_index=False)
      .mean(numeric_only=True)
)
google_avg.columns=["year","month","trend_soda_google"]

google_avg= google_avg.sort_values(["year","month"]).reset_index(drop=True)
google_avg=google_avg.iloc[1:,:]
print(google_avg.head())

#=========네이버 트랜드 변수 만들기========
# 필요한 부분만 추출
naver=pd.read_excel('/content/drive/MyDrive/프로젝트 준비/전처리/trend_soda_naver.xlsx')
naver_clean = naver.copy()
naver_clean.columns = ["date", "탄산음료"]

# 날짜 변환
naver_clean["date"] = pd.to_datetime(naver_clean["date"], errors="coerce")
naver_clean["year"] = naver_clean["date"].dt.year
naver_clean["month"] = naver_clean["date"].dt.month

naver_clean.drop(['date'], axis=1, inplace=True)

naver_clean.columns=["trend_soda_naver","year","month"]
# 결과 확인
naver_clean= naver_clean.sort_values(["year","month"]).reset_index(drop=True)
print(naver_clean.head())

#=========트랜드 변수 관련 데이터 병합==========
merged_trend = pd.merge(google_avg, naver_clean, on=["year","month"], how="outer")
merged_trend= merged_trend.sort_values(["year","month"]).reset_index(drop=True)
merged_trend

   year  month  trend_soda_google
1  2021      1              44.00
2  2021      2              47.25
3  2021      3              50.25
4  2021      4              48.00
5  2021      5              64.60
   trend_soda_naver  year  month
0          64.32020  2021      1
1          51.80163  2021      2
2          63.82800  2021      3
3          65.46248  2021      4
4          66.05683  2021      5


,year,month,trend_soda_google,trend_soda_naver
0,2021,1,44.00,64.32020
1,2021,2,47.25,51.80163
2,2021,3,50.25,63.82800
3,2021,4,48.00,65.46248
4,2021,5,64.60,66.05683
5,2021,6,49.25,68.21136
6,2021,7,69.50,68.74071
7,2021,8,43.00,63.91158
8,2021,9,53.25,70.54234
9,2021,10,45.80,62.94576


In [ ]:
merged_fin =merged.merge(merged_trend, on=["year","month"], how="left")
merged_fin

,year,month,ccsi,cpi,cpi_soda,temperature,rain,trend_soda_google,trend_soda_naver
0,2021,1,95.2,101.04,103.27,-1.1,19.9,44.00,64.32020
1,2021,2,97.4,101.58,103.79,3.4,20.1,47.25,51.80163
2,2021,3,100.6,101.84,104.72,8.7,110.7,50.25,63.82800
3,2021,4,102.5,101.98,105.91,13.2,76.3,48.00,65.46248
4,2021,5,105.7,102.05,105.22,16.6,143.8,64.60,66.05683
5,2021,6,111.1,102.05,104.55,21.7,91.6,49.25,68.21136
6,2021,7,103.6,102.26,104.63,26.0,233.8,69.50,68.74071
7,2021,8,102.9,102.75,102.58,24.8,288.4,43.00,63.91158
8,2021,9,104.2,103.17,105.66,21.3,145.8,53.25,70.54234
9,2021,10,107.4,103.35,106.31,15.1,53.9,45.80,62.94576


## 외부 데이터와 병합 진행

(1) 매입 데이터

In [ ]:
sales_fin = monthly_sales.merge(merged_fin, on=["year","month"], how="left")
sales_fin.head()

,barcode,product_name,year,month,quarter,quantity,supply_price,inbound_qty_lag_1m,inbound_qty_lag_1y,inbound_rolling_avg_3m,...,price_volatility,is_zero,volume_ml,ccsi,cpi,cpi_soda,temperature,rain,trend_soda_google,trend_soda_naver
0,18801097234229.0,(동아오츠카)오란씨파인1.5L,2021,1,1,2,24000,NaN,NaN,NaN,...,NaN,0,1500.0,95.2,101.04,103.27,-1.1,19.9,44.0,64.3202
1,18801097260020.0,(동아오츠카)데미소다애플1.5L,2021,1,1,7,72800,NaN,NaN,NaN,...,NaN,0,1500.0,95.2,101.04,103.27,-1.1,19.9,44.0,64.3202
2,18801223100268.0,(일화)맥콜페트500ml,2021,1,1,5,80000,NaN,NaN,NaN,...,NaN,0,500.0,95.2,101.04,103.27,-1.1,19.9,44.0,64.3202
3,18801223100275.0,(일화)맥콜1.5L,2021,1,1,38,585600,NaN,NaN,NaN,...,NaN,0,1500.0,95.2,101.04,103.27,-1.1,19.9,44.0,64.3202
4,18801223100503.0,(일화)맥콜캔250ml,2021,1,1,22,269000,NaN,NaN,NaN,...,NaN,0,250.0,95.2,101.04,103.27,-1.1,19.9,44.0,64.3202


(2) 매출데이터

In [ ]:
purchase_fin =monthly_purchase.merge(merged_fin, on=["year","month"], how="left")
purchase_fin.head()

,supplier_code,product_name,year,month,quarter,quantity,sales_price,holiday_count,sales_lag_1m,sales_lag_1y,...,price_volatility,is_zero,volume_ml,ccsi,cpi,cpi_soda,temperature,rain,trend_soda_google,trend_soda_naver
0,100002,(동아오츠카)데미소다애플250ml,2021,1,1,300,109000,0,NaN,NaN,...,NaN,0,250.0,95.2,101.04,103.27,-1.1,19.9,44.0,64.3202
1,100004,(롯데칠성)칠성사이다250ml,2021,1,1,6992,5609500,0,NaN,NaN,...,NaN,0,250.0,95.2,101.04,103.27,-1.1,19.9,44.0,64.3202
2,100010,(코카콜라)코카콜라1.5L,2021,1,1,4806,6458800,0,NaN,NaN,...,NaN,0,1500.0,95.2,101.04,103.27,-1.1,19.9,44.0,64.3202
3,100048,(일화)맥콜1.5L,2021,1,1,940,827100,0,NaN,NaN,...,NaN,0,1500.0,95.2,101.04,103.27,-1.1,19.9,44.0,64.3202
4,100088,(농심)웰치스청포도355ml,2021,1,1,480,781900,0,NaN,NaN,...,NaN,0,355.0,95.2,101.04,103.27,-1.1,19.9,44.0,64.3202


In [ ]:
purchase_fin

,supplier_code,product_name,year,month,quarter,quantity,sales_price,holiday_count,sales_lag_1m,sales_lag_1y,...,price_volatility,is_zero,volume_ml,ccsi,cpi,cpi_soda,temperature,rain,trend_soda_google,trend_soda_naver
0,100002,(동아오츠카)데미소다애플250ml,2021,1,1,300,109000,0,NaN,NaN,...,NaN,0,250.0,95.2,101.04,103.27,-1.1,19.9,44.0,64.32020
1,100004,(롯데칠성)칠성사이다250ml,2021,1,1,6992,5609500,0,NaN,NaN,...,NaN,0,250.0,95.2,101.04,103.27,-1.1,19.9,44.0,64.32020
2,100010,(코카콜라)코카콜라1.5L,2021,1,1,4806,6458800,0,NaN,NaN,...,NaN,0,1500.0,95.2,101.04,103.27,-1.1,19.9,44.0,64.32020
3,100048,(일화)맥콜1.5L,2021,1,1,940,827100,0,NaN,NaN,...,NaN,0,1500.0,95.2,101.04,103.27,-1.1,19.9,44.0,64.32020
4,100088,(농심)웰치스청포도355ml,2021,1,1,480,781900,0,NaN,NaN,...,NaN,0,355.0,95.2,101.04,103.27,-1.1,19.9,44.0,64.32020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
402,100010,(코카콜라)코카콜라1.5L,2024,12,4,792,662100,0,1878.0,1698.0,...,167.141622,0,1500.0,88.2,114.91,115.44,1.8,6.5,57.2,44.57652
403,100112,(롯데칠성)펩시콜라1.5L,2024,12,4,390,529300,0,1520.0,1440.0,...,275.629310,0,1500.0,88.2,114.91,115.44,1.8,6.5,57.2,44.57652
404,300022,(일화)맥콜캔250ml,2024,12,4,2300,2107100,0,2860.0,3960.0,...,91.312331,0,250.0,88.2,114.91,115.44,1.8,6.5,57.2,44.57652
405,300023,(코카콜라)코카콜라(슈퍼용)355ml,2024,12,4,300,173600,0,720.0,1260.0,...,256.084976,0,355.0,88.2,114.91,115.44,1.8,6.5,57.2,44.57652


#전처리 완료함

# 2025년 데이터 만들기

In [ ]:
def make_q4_ref_and_future(
    data: pd.DataFrame,
    key_col="product_name",
    year_col="year",
    month_col="month",
    target_cols=("quantity",),
):
    df = data.copy()

    # ---- 1) Q4 기준 레퍼런스 만들기 ----
    df["yyyymm"] = df[year_col].astype(int) * 100 + df[month_col].astype(int)
    q4 = df[(df[year_col] == 2024) & (df[month_col].isin([10, 11, 12]))].copy()
    q4 = q4.sort_values([key_col, "yyyymm"])

    time_cols = [c for c in [year_col, month_col, "day", "yyyymm"] if c in df.columns]
    exclude_cols = set([key_col] + time_cols + list(target_cols))
    cols_to_uniform = [c for c in df.columns if c not in exclude_cols]

    # Q4 최신 row 추출
    q4_latest = (
        q4.sort_values([key_col, "yyyymm"])
          .groupby(key_col)
          .last()
          .reset_index()
    )

    # product-level 속성만 ref에 포함
    ref = q4_latest[[key_col] + cols_to_uniform]

    # ---- 2) 전체 데이터에 레퍼런스 머지 ----
    merged = df.merge(ref, on=key_col, how="left", suffixes=("", "_q4ref"))
    for c in cols_to_uniform:
        merged[c] = merged[f"{c}_q4ref"].combine_first(merged[c])
        merged.drop(columns=[f"{c}_q4ref"], inplace=True)
    merged.drop(columns=["yyyymm"], errors="ignore", inplace=True)
    df_uniform = merged

    # ---- 3) 2025.01~03 skeleton 생성 ----
    items_2025 = ref[key_col].drop_duplicates()
    future_idx = pd.MultiIndex.from_product(
        [[2025], [1, 2, 3], items_2025],
        names=[year_col, month_col, key_col]
    )
    future = future_idx.to_frame(index=False)
    future = future.merge(ref, on=key_col, how="left")

    # quarter 자동 생성
    future["quarter"] = ((future[month_col] - 1) // 3 + 1)

    # 예측 대상(y) 초기화
    for c in target_cols:
        if c in df.columns:
            future[c] = np.nan

    if "day" in df.columns:
        future["day"] = 1

    return df_uniform, future

# ---------- 사용 예시 ----------
df_purchase, future_purchase = make_q4_ref_and_future(purchase_fin, key_col="product_name")
df_sales, future_sales = make_q4_ref_and_future(sales_fin, key_col="product_name")

future_purchase.drop(['ccsi', 'cpi', 'cpi_soda',
       'temperature', 'rain', 'trend_soda_google', 'trend_soda_naver'], axis=1, inplace=True)

future_sales.drop(['ccsi', 'cpi', 'cpi_soda',
       'temperature', 'rain', 'trend_soda_google', 'trend_soda_naver'], axis=1, inplace=True)

In [ ]:
future_purchase=future_purchase.merge(merged_fin, on=["year","month"], how="left")
future_sales=future_sales.merge(merged_fin, on=["year","month"], how="left")

## 1.Purchase 2025년 1월-3월 수요 예측

(1) xgboost, lightGBM 이용하여 -> 평가 지표를 활용하여 ligthGBM 이용

(2) lightGBM을 이용해서 quantity 예측

In [ ]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor, early_stopping
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error

# 1) Train/Valid split

drop_cols = ["quantity","supplier_code","product_name"]

train_1 = df_purchase[(df_purchase["year"]>=2021)&(df_purchase["year"]<=2023)].copy()
valid_1 = df_purchase[(df_purchase["year"]==2024)].copy()

X_train_1 = train_1.drop(columns=drop_cols)
y_train_1 = np.log1p(train_1["quantity"])
X_valid_1 = valid_1.drop(columns=drop_cols)
y_valid_1 = np.log1p(valid_1["quantity"])


# 2) 모델 학습 (LightGBM vs XGBoost)

# LightGBM
lgb_model = LGBMRegressor(
    objective="regression", learning_rate=0.05, num_leaves=31,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
    random_state=42, n_estimators=5000
)
lgb_model.fit(
    X_train_1, y_train_1,
    eval_set=[(X_valid_1,y_valid_1)],
    eval_metric="rmse",
    callbacks=[early_stopping(100, verbose=False)]
)

y_pred_lgb = np.expm1(lgb_model.predict(X_valid_1))
rmse_lgb = np.sqrt(mean_squared_error(valid_1["quantity"], y_pred_lgb))
mae_lgb = mean_absolute_error(valid_1["quantity"], y_pred_lgb)
print(f"[LGB] RMSE: {rmse_lgb:.4f}, MAE: {mae_lgb:.4f}")

# XGBoost
dtrain_1 = xgb.DMatrix(X_train_1, label=y_train_1)
dvalid_1 = xgb.DMatrix(X_valid_1, label=y_valid_1)
params_xgb = {
    "objective":"reg:squarederror","eval_metric":"rmse","eta":0.05,
    "max_depth":5,"subsample":0.8,"colsample_bytree":0.8,"seed":42
}
xgb_model = xgb.train(
    params_xgb, dtrain_1, num_boost_round=5000,
    evals=[(dtrain_1,"train"), (dvalid_1,"valid")],
    early_stopping_rounds=100, verbose_eval=50
)

y_pred_xgb = np.expm1(xgb_model.predict(dvalid_1))
rmse_xgb = np.sqrt(mean_squared_error(valid_1["quantity"], y_pred_xgb))
mae_xgb = mean_absolute_error(valid_1["quantity"], y_pred_xgb)
print(f"[XGB] RMSE: {rmse_xgb:.4f}, MAE: {mae_xgb:.4f}")


[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000203 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 406
[LightGBM] [Info] Number of data points in the train set: 308, number of used features: 19
[LightGBM] [Warn

In [ ]:
# 3) 최종 학습 (2021–2024 전체)
train_2 = df_purchase[(df_purchase["year"]>=2021)&(df_purchase["year"]<=2024)].copy()
X_train_2 = train_2.drop(columns=drop_cols)
y_train_2 = np.log1p(train_2["quantity"])
feature_cols = X_train_2.columns.tolist()

final_model = LGBMRegressor(
    objective="regression", learning_rate=0.05, num_leaves=31,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
    random_state=42, n_estimators=1000
)
final_model.fit(X_train_2, y_train_2)

# 4) 2025 예측

# 학습과 동일한 feature 구조 맞추기
X_future = future_purchase.drop(['product_name', 'supplier_code','quantity'], axis=1)

# 예측
future_purchase["quantity"] = np.expm1(final_model.predict(X_future))

[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000164 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 434
[LightGBM] [Info] Number of data points in 

In [ ]:
monthly_pred_purchase = (
    future_purchase.groupby(["year","month"])["quantity"]
    .sum()
    .reset_index()
)

print(monthly_pred_purchase)

   year  month      quantity
0  2025      1  13771.892299
1  2025      2  12857.590283
2  2025      3  11871.463660


## 2.Sales 2025년 1월-3월 수요 예측

(1) xgboost, lightGBM 이용하여 -> 평가 지표를 활용하여 ligthGBM 이용

(2) lightGBM을 이용해서 quantity 예측

In [ ]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor, early_stopping
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error

# 1) Train/Valid split

drop_cols = ["quantity","barcode","product_name"]

train_1 = df_sales[(df_sales["year"]>=2021)&(df_sales["year"]<=2023)].copy()
valid_1 = df_sales[(df_sales["year"]==2024)].copy()

X_train_1 = train_1.drop(columns=drop_cols)
y_train_1 = np.log1p(train_1["quantity"])
X_valid_1 = valid_1.drop(columns=drop_cols)
y_valid_1 = np.log1p(valid_1["quantity"])


# 2) 모델 학습 (LightGBM vs XGBoost)

# LightGBM
lgb_model = LGBMRegressor(
    objective="regression", learning_rate=0.05, num_leaves=31,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
    random_state=42, n_estimators=5000
)
lgb_model.fit(
    X_train_1, y_train_1,
    eval_set=[(X_valid_1,y_valid_1)],
    eval_metric="rmse",
    callbacks=[early_stopping(100, verbose=False)]
)

y_pred_lgb = np.expm1(lgb_model.predict(X_valid_1))
rmse_lgb = np.sqrt(mean_squared_error(valid_1["quantity"], y_pred_lgb))
mae_lgb = mean_absolute_error(valid_1["quantity"], y_pred_lgb)
print(f"[LGB] RMSE: {rmse_lgb:.4f}, MAE: {mae_lgb:.4f}")

# XGBoost
dtrain_1 = xgb.DMatrix(X_train_1, label=y_train_1)
dvalid_1 = xgb.DMatrix(X_valid_1, label=y_valid_1)
params_xgb = {
    "objective":"reg:squarederror","eval_metric":"rmse","eta":0.05,
    "max_depth":5,"subsample":0.8,"colsample_bytree":0.8,"seed":42
}
xgb_model = xgb.train(
    params_xgb, dtrain_1, num_boost_round=5000,
    evals=[(dtrain_1,"train"), (dvalid_1,"valid")],
    early_stopping_rounds=100, verbose_eval=50
)

y_pred_xgb = np.expm1(xgb_model.predict(dvalid_1))
rmse_xgb = np.sqrt(mean_squared_error(valid_1["quantity"], y_pred_xgb))
mae_xgb = mean_absolute_error(valid_1["quantity"], y_pred_xgb)
print(f"[XGB] RMSE: {rmse_xgb:.4f}, MAE: {mae_xgb:.4f}")


[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000509 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1824
[LightGBM] [Info] Number of data points in

In [ ]:
# 3) 최종 학습 (2021–2024 전체)
train_2 = df_sales[(df_sales["year"]>=2021)&(df_sales["year"]<=2024)].copy()
X_train_2 = train_2.drop(columns=drop_cols)
y_train_2 = np.log1p(train_2["quantity"])
feature_cols = X_train_2.columns.tolist()

final_model = LGBMRegressor(
    objective="regression", learning_rate=0.05, num_leaves=31,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
    random_state=42, n_estimators=1000
)
final_model.fit(X_train_2, y_train_2)

# 4) 2025 예측

# 학습과 동일한 feature 구조 맞추기
X_future = future_sales.drop(['product_name', 'barcode','quantity'], axis=1)

# 예측
future_sales["quantity"] = np.expm1(final_model.predict(X_future))

[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000723 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1850
[LightGBM] [Info] Number of data points in

In [ ]:
monthly_pred_sales = (
    future_sales.groupby(["year","month"])["quantity"]
    .sum()
    .reset_index()
)

print(monthly_pred_sales)

   year  month      quantity
0  2025      1  15144.877101
1  2025      2  10321.544979
2  2025      3  12950.522944


# 최종

In [ ]:
print(monthly_pred_purchase)
print(monthly_pred_sales)

   year  month      quantity
0  2025      1  13771.892299
1  2025      2  12857.590283
2  2025      3  11871.463660
   year  month      quantity
0  2025      1  15144.877101
1  2025      2  10321.544979
2  2025      3  12950.522944
